# Notebook 78: V8.9 Ring-2 Operator Freeze, Parent Equivalence and 151-Page HITL Regeneration

Fail-closed release notebook for `v8_9_operator.py`. It proves exact V8.8 parent equivalence, reproduces Notebook 75 C1, runs exact replay, and regenerates the complete 151-page HITL cohort. No clean/truth data is used by the operator.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn joblib torch torchvision pyarrow


Mounted at /content/drive


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, importlib.util, sys
import cv2, numpy as np, pandas as pd
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
OPERATOR_SOURCE=ROOT/'v8_9_release/v8_9_operator.py'
P_PRINT_BUNDLE=ROOT/'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
REMOVE_BUNDLE=ROOT/'m5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'
NB72_ROOT=ROOT/'frozen_v8_8_annotated_only_operational_trial'
NB75_ROOT=ROOT/'notebook75_collar_preservation_counterfactual'
HITL_EXPECTED_PAGES=151
RUN=ROOT/'v8_9_release'/ (datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook78')
for p in [RUN,RUN/'gates',RUN/'pages',RUN/'audit',RUN/'replay',RUN/'upload']: p.mkdir(parents=True,exist_ok=False)
def require(x,m):
 if not bool(x): raise RuntimeError(m)
def sha256_file(p):
 h=hashlib.sha256();
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
def latest(root,required):
 hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in required)]
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
require(OPERATOR_SOURCE.exists(),f'Missing {OPERATOR_SOURCE}')
spec=importlib.util.spec_from_file_location('v8_9_operator',OPERATOR_SOURCE);
mod=importlib.util.module_from_spec(spec);
sys.modules['v8_9_operator']=mod
spec.loader.exec_module(mod)
op=mod.load_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE)
print('Run:',RUN)


## Gate 1: exact V8.8 parent equivalence

In [ ]:
NB72=latest(NB72_ROOT,['page_roster.csv'])
require(NB72 is not None,'Complete Notebook 72 run not found.')
roster=pd.read_csv(NB72/'page_roster.csv')
required={'page','observed_path','seed_path'}; require(required<=set(roster.columns),f'Notebook72 roster missing {required-set(roster.columns)}')
rows=[]
for _,r in roster.iterrows():
 obs=cv2.imread(str(r.observed_path),0); seed=cv2.imread(str(r.seed_path),0)
 require(obs is not None and seed is not None and obs.shape==seed.shape,f'{r.page}: input failure')
 parent=mod.V88Operator(P_PRINT_BUNDLE,REMOVE_BUNDLE).run_page(obs,seed,page=str(r.page))
 child=op.run_page(obs,seed,page=str(r.page))
 ring2=child.ring2_mask
 rows.append({'page':r.page,'parent_non_ring2_mismatch':int(np.sum(parent.corrected[~ring2]!=child.corrected[~ring2])),
              'parent_arrays_equal':all(np.array_equal(getattr(parent,k),getattr(child,k)) for k in ['observed','authoritative_mask','baseline_v78','m3_mask','m5_remove_mask','safe_remove_mask','p_print','p_remove','reason_map']),
              'ring2_pixels':int(ring2.sum()),'delta_pixels':int(child.v89_delta_mask.sum())})
eq=pd.DataFrame(rows); eq.to_csv(RUN/'gates'/'v88_parent_equivalence.csv',index=False)
require((eq.parent_non_ring2_mismatch==0).all() and eq.parent_arrays_equal.all(),'V8.8 parent-equivalence gate failed.')
print('PASS: V8.8 parent equivalence',len(eq))


## Gate 2: Notebook 75 C1 exact reproduction

In [ ]:
NB75=latest(NB75_ROOT,['tables/page_metrics.csv'])
require(NB75 is not None,'Complete Notebook 75 run not found.')
ref=pd.read_csv(NB75/'tables'/'page_metrics.csv'); require('page' in ref.columns,'Notebook75 page metrics lacks page')
c1=[]
for _,r in roster.iterrows():
 obs=cv2.imread(str(r.observed_path),0); seed=cv2.imread(str(r.seed_path),0); z=op.run_page(obs,seed,page=str(r.page))
 c1_path=NB75/'pages'/str(r.page)/'c1_preserve_ring2.png'
 require(c1_path.exists(),f'{r.page}: Notebook75 C1 page missing')
 expected=cv2.imread(str(c1_path),0); require(expected is not None and expected.shape==z.corrected.shape,f'{r.page}: C1 unreadable/shape')
 c1.append({'page':r.page,'different_pixels':int(np.sum(expected!=z.corrected)),'expected_sha256':sha256_file(c1_path)})
c1=pd.DataFrame(c1); c1.to_csv(RUN/'gates'/'notebook75_c1_reproduction.csv',index=False)
require((c1.different_pixels==0).all(),'Notebook75 C1 reproduction failed.')
print('PASS: Notebook75 C1 reproduction',len(c1))


## Gate 3: exact deterministic replay

In [ ]:
sample=roster.sort_values('page').head(min(12,len(roster))); replay=[]
for _,r in sample.iterrows():
 obs=cv2.imread(str(r.observed_path),0); seed=cv2.imread(str(r.seed_path),0)
 a=op.run_page(obs,seed,page=str(r.page)); b=op.run_page(obs,seed,page=str(r.page))
 replay.append({'page':r.page,'corrected_exact':np.array_equal(a.corrected,b.corrected),'ring2_exact':np.array_equal(a.ring2_mask,b.ring2_mask),'p_print_exact':np.array_equal(a.p_print,b.p_print,equal_nan=True),'p_remove_exact':np.array_equal(a.p_remove,b.p_remove,equal_nan=True)})
replay=pd.DataFrame(replay); replay.to_csv(RUN/'replay'/'replay_results.csv',index=False)
require(replay.drop(columns='page').all().all(),'Replay gate failed.')
print('PASS: exact replay')


## Gate 4: full 151-page HITL regeneration

In [ ]:
require(len(roster)==HITL_EXPECTED_PAGES,f'Expected 151 HITL pages, found {len(roster)}')
page_rows=[]
for _,r in roster.iterrows():
 page=str(r.page); obs=cv2.imread(str(r.observed_path),0); seed=cv2.imread(str(r.seed_path),0); z=op.run_page(obs,seed,page=page)
 d=RUN/'pages'/page; d.mkdir(parents=True,exist_ok=False)
 cv2.imwrite(str(d/'observed.png'),z.observed); cv2.imwrite(str(d/'authoritative_seed_mask.png'),seed); cv2.imwrite(str(d/'v8_8_parent.png'),z.parent_v88_corrected); cv2.imwrite(str(d/'v8_9.png'),z.corrected); cv2.imwrite(str(d/'ring2_mask.png'),z.ring2_mask.astype(np.uint8)*255); cv2.imwrite(str(d/'v8_9_delta.png'),z.v89_delta_mask.astype(np.uint8)*255)
 (d/'page_audit.json').write_text(json.dumps(z.audit,indent=2,default=str))
 page_rows.append({'page':page,'ring2_pixels':int(z.ring2_mask.sum()),'delta_pixels':int(z.v89_delta_mask.sum()),'non_ring2_parent_mismatch':int(np.sum(z.corrected[~z.ring2_mask]!=z.parent_v88_corrected[~z.ring2_mask])),'v89_sha256':sha256_file(d/'v8_9.png')})
pages=pd.DataFrame(page_rows); pages.to_csv(RUN/'audit'/'151_page_regeneration.csv',index=False)
require(len(pages)==151 and (pages.non_ring2_parent_mismatch==0).all(),'151-page regeneration gate failed.')
print('PASS: regenerated',len(pages),'HITL pages')


## Release verdict and upload package

In [ ]:
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.9','verdict':'PASS','gates':{'v8_8_parent_equivalence':True,'notebook75_c1_reproduction':True,'exact_replay':True,'hitl_151_page_regeneration':True},'pages':151,'operator_sha256':sha256_file(OPERATOR_SOURCE),'uses_clean_truth':False}
(RUN/'release_verdict.json').write_text(json.dumps(verdict,indent=2))
shutil.copy2(OPERATOR_SOURCE,RUN/'upload'/'v8_9_operator.py'); shutil.copy2(RUN/'release_verdict.json',RUN/'upload'/'release_verdict.json'); shutil.copy2(RUN/'audit'/'151_page_regeneration.csv',RUN/'upload'/'151_page_regeneration.csv'); shutil.copy2(RUN/'gates'/'v88_parent_equivalence.csv',RUN/'upload'/'v88_parent_equivalence.csv'); shutil.copy2(RUN/'gates'/'notebook75_c1_reproduction.csv',RUN/'upload'/'notebook75_c1_reproduction.csv'); shutil.make_archive(str(RUN/'notebook78_upload_bundle'),'zip',RUN/'upload')
print(json.dumps(verdict,indent=2)); print('COMPLETE:',RUN)
